In [1]:
import sqlite3
import pandas as pd
import numpy as np

print("[*] Connecting to database...")
conn = sqlite3.connect('../m5_forecasting.db')

# 1. SQL Heavy Lifting: Extracting daily aggregated sales
query = """
    SELECT 
        c.calendar_date,
        SUM(s.units_sold) AS total_daily_sales
    FROM fact_sales s
    JOIN dim_calendar c ON s.date_id = c.date_id
    GROUP BY c.calendar_date
    ORDER BY c.calendar_date ASC;
"""

print("[*] Querying 58.5 million rows for daily aggregates. This will take ~60 seconds...")
df = pd.read_sql(query, conn)
conn.close()

# 2. Data Preparation
df['calendar_date'] = pd.to_datetime(df['calendar_date'])
df.set_index('calendar_date', inplace=True)

# 3. The 30-Day Moving Average Baseline
print("[*] Calculating 30-Day Moving Average...")
df['baseline_forecast_30d'] = df['total_daily_sales'].rolling(window=30).mean().shift(1)

# 4. Drop the first 30 days (since they can't have a 30-day historical average)
df_eval = df.dropna()

# 5. Evaluate the Baseline (MAPE & RMSE)
# RMSE (Root Mean Squared Error)
rmse = np.sqrt(np.mean((df_eval['total_daily_sales'] - df_eval['baseline_forecast_30d'])**2))

# MAPE (Mean Absolute Percentage Error)
mape = np.mean(np.abs((df_eval['total_daily_sales'] - df_eval['baseline_forecast_30d']) / df_eval['total_daily_sales'])) * 100

print("\n=== BASELINE MODEL METRICS ===")
print(f"[-] RMSE: {rmse:.2f} units")
print(f"[-] MAPE: {mape:.2f}%")
print("===============================\n")

# Preview the final data
display(df_eval.tail(10))

[*] Connecting to database...
[*] Querying 58.5 million rows for daily aggregates. This will take ~60 seconds...
[*] Calculating 30-Day Moving Average...

=== BASELINE MODEL METRICS ===
[-] RMSE: 5824.00 units
[-] MAPE: 609.99%



,total_daily_sales,baseline_forecast_30d
calendar_date,,
2016-04-15,41789,41642.900000
2016-04-16,48362,41815.666667
2016-04-17,51640,42260.766667
2016-04-18,38059,42660.366667
2016-04-19,37570,42389.633333
2016-04-20,35343,42047.800000
2016-04-21,35033,41980.566667
2016-04-22,40517,41965.833333
2016-04-23,48962,42156.866667


In [2]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import warnings

# Suppress harmless statistical warnings for clean output
warnings.filterwarnings("ignore")

print("[*] Training Advanced Model (Holt-Winters Exponential Smoothing)...")

# 1. Train the model (accounting for weekly 7-day seasonality and overall trend)
model = ExponentialSmoothing(
    df['total_daily_sales'], 
    trend='add', 
    seasonal='add', 
    seasonal_periods=7
).fit()

# 2. Generate the forecast
df['hw_forecast'] = model.fittedvalues

# 3. Evaluate on the exact same timeframe as the baseline for a 100% fair comparison
df_eval_ml = df.dropna()

# 4. Calculate New RMSE
rmse_ml = np.sqrt(np.mean((df_eval_ml['total_daily_sales'] - df_eval_ml['hw_forecast'])**2))

print("\n=== ADVANCED MODEL METRICS ===")
print(f"[-] Baseline RMSE: 5824.00 units")
print(f"[-] Advanced RMSE: {rmse_ml:.2f} units")
print(f"[-] Error Reduction: {((5824.00 - rmse_ml) / 5824.00) * 100:.2f}%")
print("==============================\n")

# Preview the final comparison data
display(df_eval_ml[['total_daily_sales', 'baseline_forecast_30d', 'hw_forecast']].tail(10))

[*] Training Advanced Model (Holt-Winters Exponential Smoothing)...

=== ADVANCED MODEL METRICS ===
[-] Baseline RMSE: 5824.00 units
[-] Advanced RMSE: 3296.99 units
[-] Error Reduction: 43.39%



,total_daily_sales,baseline_forecast_30d,hw_forecast
calendar_date,,,
2016-04-15,41789,41642.900000,41597.728240
2016-04-16,48362,41815.666667,51371.605430
2016-04-17,51640,42260.766667,49870.108533
2016-04-18,38059,42660.366667,40198.592660
2016-04-19,37570,42389.633333,37038.769159
2016-04-20,35343,42047.800000,36942.042401
2016-04-21,35033,41980.566667,37053.460129
2016-04-22,40517,41965.833333,39628.420310
2016-04-23,48962,42156.866667,49489.511397


In [3]:
# Exporting the final forecast data for Tableau visualization
print("[*] Exporting forecast results to CSV...")
df_eval_ml[['total_daily_sales', 'baseline_forecast_30d', 'hw_forecast']].to_csv('../data_processed/final_forecast_results.csv')
print("[+] Export complete: 'final_forecast_results.csv' safely stored in /data_processed/")

[*] Exporting forecast results to CSV...
[+] Export complete: 'final_forecast_results.csv' safely stored in /data_processed/
